In [ ]:
# ══════════════════════════════════════════════════════════════════
# 7. PillServiceImpl 검색 시뮬레이션
#    - 필터 단계별 후보 수 분포
#    - 기존(strict) / relaxed / improved 정확도 비교
# ══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from rapidfuzz.distance import Levenshtein

sim_df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
print(f"로드: {len(sim_df)}개\n")


# ── 1. Java 필터 로직 재현 ─────────────────────────────────────────

COLOR_CANDIDATE_IDS = {
    "white":       {1,17,31,34,35,38,39,41,51,52,14,20,36},
    "gray":        {14,20,36,1,17,31,34,35,38,39,41,51,52},
    "light_green": {7,27,50,8,32,35,38,46},
    "green":       {8,32,35,38,46,7,27,50},
    "pink":        {4,44,45,3,23,39,43},
    "orange":      {3,23,39,43,2,24,28,41,42,6,33,34,4,44,45},
    "yellow":      {2,24,28,41,42,3,23,39,43},
    "brown":       {6,33,34,3,23,39,43},
    "blue":        {10,18,19,29,51,1,17,31,34,35,38,39,41,52},
    "teal":        {9,21,40,52},
    "navy":        {11,25},
    "purple":      {13,22,26,37,47,48},
    "red":         {5,30},
    "black":       {15,49},
    "transparent": {16,19,21,24,26,31,38,39,48,49,50},
}

SHAPE_IDS = {
    "oval":      {2,3}, "oblong": {3,2}, "capsule": {3,2},
    "round":     {1,2}, "triangle": {5}, "rectangle": {6},
    "diamond":   {7},   "pentagon": {8}, "hexagon":   {9}, "octagon": {10},
}

CONFUSION_TABLE = str.maketrans({"0":"O","1":"I","L":"I","5":"S","Λ":"A","α":"A","T":"1"})

def normalize(s: str) -> str:
    return str(s).strip().lower().replace(" ", "").replace("-", "")

def expand_imprint(text: str) -> list[str]:
    variants = [text]
    upper = (text.upper()
             .replace("®", "").replace("α", "A").replace("Λ", "A"))
    clean = upper.replace(" ", "")
    if len(clean) <= 3:
        variants.append(clean.translate(CONFUSION_TABLE))
    variants.append(upper.replace(" ", ""))
    return list(dict.fromkeys(variants))  # 순서 유지 dedup

def preprocess_texts(texts) -> list[str]:
    if not isinstance(texts, list):
        texts = [str(texts)] if pd.notna(texts) else []
    result = []
    for t in texts:
        t = str(t).strip()
        if not t or t in ("nan", ""):
            continue
        if "마크" in t or "로고" in t:
            continue
        result.extend(expand_imprint(t))
    return list(dict.fromkeys(result))

def is_print_match_improved(pred, gt) -> bool:
    gt_c = "" if str(gt).strip() in ("-","nan","") else str(gt).strip()
    pred_c = "" if str(pred).strip() in ("nan","") else str(pred).strip()
    if not gt_c:
        return pred_c == ""
    if "마크" in gt_c or "로고" in gt_c:
        return True                            # 마크는 스킵으로 간주 → pass
    if len(gt_c.replace(" ", "")) <= 1:
        return True                            # 1글자 스킵 → pass
    if normalize(gt_c) == normalize(pred_c):
        return True
    # confusion 정규화 후 비교
    def norm_confusion(s):
        return s.upper().replace("®","").replace("α","A").replace("Λ","A").replace(" ","")
    if norm_confusion(gt_c) == norm_confusion(pred_c):
        return True
    # edit distance ≤ 1
    return Levenshtein.distance(normalize(gt_c), normalize(pred_c)) <= 1


# ── 2. 필터 단계별 후보 수 시뮬레이션 ────────────────────────────────
# CSV에는 실제 DB가 없으므로, 각 필터 통과 여부를 정답 컬럼으로 근사

def simulate_filter_stage(row):
    """
    텍스트 검색 후보 수를 직접 알 수 없으므로
    각 필터(color, shape)가 정답을 유지하는지로 단계 추적
    - stage0: 필터 없음 (텍스트 검색만)
    - stage1: shape 필터 후
    - stage2: color 필터 후
    """
    # print 매칭 여부 (개선 기준)
    print_ok  = is_print_match_improved(row["pred_print_front"], row["gt_print_front"])
    shape_ok  = bool(row["correct_shape_relaxed"])
    color_ok  = bool(row["correct_color_relaxed"])

    return pd.Series({
        "print_ok":  print_ok,
        "shape_ok":  shape_ok,
        "color_ok":  color_ok,
        "text_only":         print_ok,
        "text_shape":        print_ok and shape_ok,
        "text_shape_color":  print_ok and shape_ok and color_ok,
    })

stage = sim_df.apply(simulate_filter_stage, axis=1)
sim_df = pd.concat([sim_df, stage], axis=1)


# ── 3. 정확도 비교표 ──────────────────────────────────────────────
print("=" * 60)
print(f"{'지표':<28} {'strict':>8} {'relaxed':>9} {'improved':>10}")
print("=" * 60)

rows = [
    ("color 정확도",   "correct_color",       "correct_color_relaxed",  "color_ok"),
    ("shape 정확도",   "correct_shape",       "correct_shape_relaxed",  "shape_ok"),
    ("print 정확도",   "correct_print_front", "correct_print_front",    "print_ok"),
    ("전체 일치",      "all_correct",         "all_correct_relaxed",    "text_shape_color"),
]
for label, s_col, r_col, i_col in rows:
    s = sim_df[s_col].mean()
    r = sim_df[r_col].mean()
    i = sim_df[i_col].mean()
    print(f"{label:<28} {s:>7.1%} {r:>9.1%} {i:>10.1%}")
print("=" * 60)


# ── 4. 필터 단계별 통과율 ─────────────────────────────────────────
print("\n[ 필터 단계별 누적 통과율 ]")
print(f"  텍스트만          : {sim_df['text_only'].mean():.1%}  ({sim_df['text_only'].sum()}건)")
print(f"  텍스트 + shape    : {sim_df['text_shape'].mean():.1%}  ({sim_df['text_shape'].sum()}건)")
print(f"  텍스트 + shape + color : {sim_df['text_shape_color'].mean():.1%}  ({sim_df['text_shape_color'].sum()}건)")


# ── 5. print 개선 상세 ────────────────────────────────────────────
sim_df["correct_print_improved"] = sim_df["print_ok"]
n_fixed  = (~sim_df["correct_print_front"] &  sim_df["correct_print_improved"]).sum()
n_broken = ( sim_df["correct_print_front"] & ~sim_df["correct_print_improved"]).sum()
print(f"\n[ print 개선 효과 ]")
print(f"  새로 맞은 케이스 : +{n_fixed}건")
print(f"  새로 틀린 케이스 : -{n_broken}건  ← 0이어야 정상")

# 새로 맞은 케이스 샘플
newly = sim_df[~sim_df["correct_print_front"] & sim_df["correct_print_improved"]]
print(f"\n[ 새로 맞은 케이스 샘플 ]")
print(newly[["gt_print_front","pred_print_front"]].head(20).to_string())


# ── 6. 아직 틀리는 케이스 분류 ───────────────────────────────────
still_wrong = sim_df[~sim_df["text_shape_color"]]
print(f"\n[ 개선 후에도 틀리는 케이스: {len(still_wrong)}건 ]")

# 원인별 분류
only_color  = (~sim_df["color_ok"] &  sim_df["shape_ok"] &  sim_df["print_ok"])
only_shape  = ( sim_df["color_ok"] & ~sim_df["shape_ok"] &  sim_df["print_ok"])
only_print  = ( sim_df["color_ok"] &  sim_df["shape_ok"] & ~sim_df["print_ok"])
multi_wrong = (~sim_df["color_ok"] | ~sim_df["shape_ok"] | ~sim_df["print_ok"]) & ~only_color & ~only_shape & ~only_print

print(f"  color만 틀림  : {only_color.sum()}건")
print(f"  shape만 틀림  : {only_shape.sum()}건")
print(f"  print만 틀림  : {only_print.sum()}건")
print(f"  복합 오류     : {multi_wrong.sum()}건")

# print만 틀린 케이스 상세
print(f"\n[ print만 틀린 케이스 상위 20 ]")
print(sim_df[only_print][["gt_print_front","pred_print_front"]].head(20).to_string())